In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_teddynote.korean import stopwords

stopword = stopwords()
stopword[:20]

['아',
 '휴',
 '아이구',
 '아이쿠',
 '아이고',
 '어',
 '나',
 '우리',
 '저희',
 '따라',
 '의해',
 '을',
 '를',
 '에',
 '의',
 '가',
 '으로',
 '로',
 '에게',
 '뿐이다']

In [3]:
# !uv add pymupdf

In [30]:
import os
import glob
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_teddynote.community.pinecone import preprocess_documents
from langchain_teddynote.community.pinecone import (
    create_index, create_sparse_encoder, fit_sparse_encoder, load_sparse_encoder, upsert_documents, upsert_documents_parallel,
    delete_namespace, init_pinecone_index, PineconeKiwiHybridRetriever
)
from langchain_openai import OpenAIEmbeddings

In [5]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
split_docs = []

files = sorted(glob.glob("../data/*.pdf"))

for file in files:
    loader = PyMuPDFLoader(file)
    split_docs.extend(loader.load_and_split(text_splitter))

    len(split_docs)

In [6]:
split_docs[0].metadata

{'producer': 'Hancom PDF 1.3.0.542',
 'creator': 'Hwp 2018 10.0.0.13462',
 'creationdate': '2023-12-08T13:28:38+09:00',
 'source': '../data\\SPRI_AI_Brief_2023년12월호_F.pdf',
 'file_path': '../data\\SPRI_AI_Brief_2023년12월호_F.pdf',
 'total_pages': 23,
 'format': 'PDF 1.4',
 'title': '',
 'author': 'dj',
 'subject': '',
 'keywords': '',
 'moddate': '2023-12-08T13:28:38+09:00',
 'trapped': '',
 'modDate': "D:20231208132838+09'00'",
 'creationDate': "D:20231208132838+09'00'",
 'page': 0}

In [7]:
contents, metadatas = preprocess_documents(
    split_docs=split_docs,
    metadata_keys=["source", "page", "author"],
    min_length=5,
    use_basename=True,
)

100%|██████████| 119/119 [00:00<00:00, 103445.01it/s]


In [8]:
contents[:5]

['2023년 12월호',
 '2023년 12월호\nⅠ. 인공지능 산업 동향 브리프\n 1. 정책/법제 \n   ▹ 미국, 안전하고 신뢰할 수 있는 AI 개발과 사용에 관한 행정명령 발표  ························· 1\n   ▹ G7, 히로시마 AI 프로세스를 통해 AI 기업 대상 국제 행동강령에 합의··························· 2\n   ▹ 영국 AI 안전성 정상회의에 참가한 28개국, AI 위험에 공동 대응 선언··························· 3',
 '▹ 미국 법원, 예술가들이 생성 AI 기업에 제기한 저작권 소송 기각····································· 4\n   ▹ 미국 연방거래위원회, 저작권청에 소비자 보호와 경쟁 측면의 AI 의견서 제출················· 5\n   ▹ EU AI 법 3자 협상, 기반모델 규제 관련 견해차로 난항··················································· 6\n \n 2. 기업/산업',
 '2. 기업/산업 \n   ▹ 미국 프런티어 모델 포럼, 1,000만 달러 규모의 AI 안전 기금 조성································ 7\n   ▹ 코히어, 데이터 투명성 확보를 위한 데이터 출처 탐색기 공개  ······································· 8\n   ▹ 알리바바 클라우드, 최신 LLM ‘통이치엔원 2.0’ 공개 ······················································ 9',
 '▹ 삼성전자, 자체 개발 생성 AI ‘삼성 가우스’ 공개 ··························································· 10\n   ▹ 구글, 앤스로픽에 20억 달러 투자로 생성 AI 협력 강화 ···········································

In [9]:
metadatas["source"][:5]

['SPRI_AI_Brief_2023년12월호_F.pdf',
 'SPRI_AI_Brief_2023년12월호_F.pdf',
 'SPRI_AI_Brief_2023년12월호_F.pdf',
 'SPRI_AI_Brief_2023년12월호_F.pdf',
 'SPRI_AI_Brief_2023년12월호_F.pdf']

In [10]:
len(contents), len(metadatas["source"]), len(metadatas["page"])

(119, 119, 119)

In [11]:
pc_index = create_index(
    api_key=os.environ["PINECONE_API_KEY"],
    index_name="teddynote-db-index",
    dimension=3072,
    metric="dotproduct",
)

[create_index]
{'dimension': 3072,
 'index_fullness': 0.0,
 'namespaces': {'teddynote-namespace-02': {'vector_count': 119}},
 'total_vector_count': 119}


In [12]:
sparse_encoder = create_sparse_encoder(stopwords(), mode="kiwiz")

In [13]:
saved_path = fit_sparse_encoder(
    sparse_encoder=sparse_encoder,
    contents=contents,
    save_path="./sparse_encoder.pkl"
)

100%|██████████| 119/119 [00:00<00:00, 1566.59it/s]

[fit_sparse_encoder]
Saved Sparse Encoder to: ./sparse_encoder.pkl


In [14]:
sparse_encoder = load_sparse_encoder("./sparse_encoder.pkl")

[load_sparse_encoder]
Loaded Sparse Encoder from: ./sparse_encoder.pkl


In [15]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [23]:
%%time
upsert_documents(
    index=pc_index,
    namespace="teddynote-namespace-01",
    contents=contents,
    metadatas=metadatas,
    sparse_encoder=sparse_encoder,
    embedder=embeddings,
    batch_size=32,
)

100%|██████████| 4/4 [00:06<00:00,  1.70s/it]


[upsert_documents]
{'dimension': 3072,
 'index_fullness': 0.0,
 'namespaces': {'teddynote-namespace-01': {'vector_count': 119}},
 'total_vector_count': 119}
CPU times: total: 312 ms
Wall time: 7.06 s


In [24]:
%%time
upsert_documents_parallel(
    index=pc_index,
    namespace="teddynote-namespace-02",
    contents=contents,
    metadatas=metadatas,
    sparse_encoder=sparse_encoder,
    embedder=embeddings,
    batch_size=512,
    max_workers=30,
)

문서 Upsert 중: 100%|██████████| 1/1 [00:02<00:00,  2.94s/it]


총 119개의 Vector 가 Upsert 되었습니다.
{'dimension': 3072,
 'index_fullness': 0.0,
 'namespaces': {'teddynote-namespace-01': {'vector_count': 119},
                'teddynote-namespace-02': {'vector_count': 119}},
 'total_vector_count': 238}
CPU times: total: 219 ms
Wall time: 3.2 s


In [25]:
pc_index.describe_index_stats()

{'dimension': 3072,
 'index_fullness': 0.0,
 'namespaces': {'teddynote-namespace-01': {'vector_count': 119},
                'teddynote-namespace-02': {'vector_count': 119}},
 'total_vector_count': 238}

In [26]:
delete_namespace(
    pinecone_index=pc_index,
    namespace="teddynote-namespace-01",
)

네임스페이스 'teddynote-namespace-01'의 모든 데이터가 삭제되었습니다.


In [27]:
pc_index.describe_index_stats()

{'dimension': 3072,
 'index_fullness': 0.0,
 'namespaces': {'teddynote-namespace-02': {'vector_count': 119}},
 'total_vector_count': 119}

In [29]:
pinecone_params = init_pinecone_index(
    index_name="teddynote-db-index",
    namespace="teddynote-namespace-02",
    api_key=os.environ["PINECONE_API_KEY"],
    sparse_encoder_path="./sparse_encoder.pkl",
    stopwords=stopwords(),
    tokenizer="kiwi",
    embeddings=embeddings,
    top_k=5,
    alpha=0.5
)

[init_pinecone_index]
{'dimension': 3072,
 'index_fullness': 0.0,
 'namespaces': {'teddynote-namespace-02': {'vector_count': 119}},
 'total_vector_count': 119}


In [31]:
pinecone_retriever = PineconeKiwiHybridRetriever(**pinecone_params)

In [32]:
search_results = pinecone_retriever.invoke("gpt-4o 미니 출시 관련 정보에 대해서 알려줘")
for result in search_results:
    print(result.page_content)
    print(result.metadata)
    print("\n====================\n")

▹ 구글 딥마인드, 범용 AI 모델의 기능과 동작에 대한 분류 체계 발표······························ 16
   ▹ 갈릴레오의 LLM 환각 지수 평가에서 GPT-4가 가장 우수 ··········································· 17
   
 4. 인력/교육     
   ▹ 영국 옥스퍼드 인터넷 연구소, AI 기술자의 임금이 평균 21% 높아······························· 18
   
   
 
Ⅱ. 주요 행사
{'context': '▹ 구글 딥마인드, 범용 AI 모델의 기능과 동작에 대한 분류 체계 발표······························ 16\n   ▹ 갈릴레오의 LLM 환각 지수 평가에서 GPT-4가 가장 우수 ··········································· 17\n   \n 4. 인력/교육     \n   ▹ 영국 옥스퍼드 인터넷 연구소, AI 기술자의 임금이 평균 21% 높아······························· 18\n   \n   \n \nⅡ. 주요 행사', 'page': 1.0, 'author': 'dj', 'source': 'SPRI_AI_Brief_2023년12월호_F.pdf'}


1. 정책/법제  
2. 기업/산업 
3. 기술/연구 
 4. 인력/교육
갈릴레오의 LLM 환각 지수 평가에서 GPT-4가 가장 우수
n 주요 LLM의 환각 현상을 평가한 ‘LLM 환각 지수’에 따르면 GPT-4는 작업 유형과 관계없이 
가장 우수한 성능을 보였으며 GPT-3.5도 거의 동등한 성능을 발휘
n 오픈소스 모델 중에서는 메타의 라마2가 RAG 없는 질문과 답변 및 긴 형식의 텍스트 
생성에서 가장 우수한 성능을 발휘
KEY Contents
{'context': '1. 정책/법제  \n2. 기업/산업 \n3. 기술/연구 \n 4. 인력/교육\n갈릴레오의 LLM 환각 지수 평가에서 GPT-4가 가장 우수\nn 주

In [33]:
search_results = pinecone_retriever.invoke(
    "gpt-4o 미니 출시 관련 정보에 대해서 알려줘", search_kwargs={"k": 1}
)
for result in search_results:
    print(result.page_content)
    print(result.metadata)
    print("\n====================\n")

▹ 구글 딥마인드, 범용 AI 모델의 기능과 동작에 대한 분류 체계 발표······························ 16
   ▹ 갈릴레오의 LLM 환각 지수 평가에서 GPT-4가 가장 우수 ··········································· 17
   
 4. 인력/교육     
   ▹ 영국 옥스퍼드 인터넷 연구소, AI 기술자의 임금이 평균 21% 높아······························· 18
   
   
 
Ⅱ. 주요 행사
{'context': '▹ 구글 딥마인드, 범용 AI 모델의 기능과 동작에 대한 분류 체계 발표······························ 16\n   ▹ 갈릴레오의 LLM 환각 지수 평가에서 GPT-4가 가장 우수 ··········································· 17\n   \n 4. 인력/교육     \n   ▹ 영국 옥스퍼드 인터넷 연구소, AI 기술자의 임금이 평균 21% 높아······························· 18\n   \n   \n \nⅡ. 주요 행사', 'page': 1.0, 'author': 'dj', 'source': 'SPRI_AI_Brief_2023년12월호_F.pdf'}




In [34]:
search_results = pinecone_retriever.invoke(
    "앤스로픽", search_kwargs={"alpha": 1, "k": 1}
)

for result in search_results:
    print(result.page_content)
    print(result.metadata)
    print("\n====================\n")

£ 구글, 앤스로픽에 최대 20억 달러 투자 합의 및 클라우드 서비스 제공
n 구글이 2023년 10월 27일 앤스로픽에 최대 20억 달러를 투자하기로 합의했으며, 이 중 5억 
달러를 우선 투자하고 향후 15억 달러를 추가로 투자할 방침
∙구글은 2023년 2월 앤스로픽에 이미 5억 5,000만 달러를 투자한 바 있으며, 아마존도 지난 9월 
앤스로픽에 최대 40억 달러의 투자 계획을 공개
∙한편, 2023년 11월 8일 블룸버그 보도에 따르면 앤스로픽은 구글의 클라우드 서비스 사용을 위해
{'context': '£ 구글, 앤스로픽에 최대 20억 달러 투자 합의 및 클라우드 서비스 제공\nn 구글이 2023년 10월 27일 앤스로픽에 최대 20억 달러를 투자하기로 합의했으며, 이 중 5억 \n달러를 우선 투자하고 향후 15억 달러를 추가로 투자할 방침\n∙구글은 2023년 2월 앤스로픽에 이미 5억 5,000만 달러를 투자한 바 있으며, 아마존도 지난 9월 \n앤스로픽에 최대 40억 달러의 투자 계획을 공개\n∙한편, 2023년 11월 8일 블룸버그 보도에 따르면 앤스로픽은 구글의 클라우드 서비스 사용을 위해', 'page': 13.0, 'author': 'dj', 'source': 'SPRI_AI_Brief_2023년12월호_F.pdf'}




In [36]:
search_results = pinecone_retriever.invoke(
    "앤스로픽", search_kwargs={"alpha": 0, "k": 1}
)
for result in search_results:
    print(result.page_content)
    print(result.metadata)
    print("\n====================\n")

모델 평가 기법 개발에 자금을 중점 지원할 계획
KEY Contents
£ 프런티어 모델 포럼, 자선단체와 함께 AI 안전 연구를 위한 기금 조성
n 구글, 앤스로픽, 마이크로소프트, 오픈AI가 출범한 프런티어 모델 포럼이 2023년 10월 25일 AI 안전 
연구를 위한 기금을 조성한다고 발표
∙참여사들은 맥거번 재단(Patrick J. McGovern Foundation), 데이비드 앤 루실 패커드 재단(The
{'context': '모델 평가 기법 개발에 자금을 중점 지원할 계획\nKEY Contents\n£ 프런티어 모델 포럼, 자선단체와 함께 AI 안전 연구를 위한 기금 조성\nn 구글, 앤스로픽, 마이크로소프트, 오픈AI가 출범한 프런티어 모델 포럼이 2023년 10월 25일 AI 안전 \n연구를 위한 기금을 조성한다고 발표\n∙참여사들은 맥거번 재단(Patrick J. McGovern Foundation), 데이비드 앤 루실 패커드 재단(The', 'page': 9.0, 'author': 'dj', 'source': 'SPRI_AI_Brief_2023년12월호_F.pdf'}




In [37]:
search_results = pinecone_retriever.invoke(
    "앤스로픽의 claude 출시 관련 내용을 알려줘",
    search_kwargs={"filter": {"page": {"$lt": 5}}, "k": 2},
)
for result in search_results:
    print(result.page_content)
    print(result.metadata)
    print("\n====================\n")

▹ 미국 법원, 예술가들이 생성 AI 기업에 제기한 저작권 소송 기각····································· 4
   ▹ 미국 연방거래위원회, 저작권청에 소비자 보호와 경쟁 측면의 AI 의견서 제출················· 5
   ▹ EU AI 법 3자 협상, 기반모델 규제 관련 견해차로 난항··················································· 6
 
 2. 기업/산업
{'context': '▹ 미국 법원, 예술가들이 생성 AI 기업에 제기한 저작권 소송 기각····································· 4\n   ▹ 미국 연방거래위원회, 저작권청에 소비자 보호와 경쟁 측면의 AI 의견서 제출················· 5\n   ▹ EU AI 법 3자 협상, 기반모델 규제 관련 견해차로 난항··················································· 6\n \n 2. 기업/산업', 'page': 1.0, 'author': 'dj', 'source': 'SPRI_AI_Brief_2023년12월호_F.pdf'}


▹ 삼성전자, 자체 개발 생성 AI ‘삼성 가우스’ 공개 ··························································· 10
   ▹ 구글, 앤스로픽에 20억 달러 투자로 생성 AI 협력 강화 ················································ 11
   ▹ IDC, 2027년 AI 소프트웨어 매출 2,500억 달러 돌파 전망··········································· 12
{'context': '▹ 삼성전자, 자체 개발 생성 AI ‘삼성 가우스’ 공개 ··························································· 10\n   ▹ 구글, 앤스로픽에 20억 달러 투자로

In [40]:
search_results = pinecone_retriever.invoke(
    "앤스로픽의 claude 3.5 출시 관련 내용을 알려줘",
    search_kwargs={
        "filter": {"source": {"$eq": "SPRi AI Brief_8월호_산업동향.pdf"}},
        "k": 3,
    }
)
for result in search_results:
    print(result.page_content)
    print(result.metadata)
    print("\n====================\n")